In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Ensure 'base' is defined from previous cells (e.g., cell 85c283ac)
if 'base' not in globals():
    print("Error: 'base' is not defined. Please run the cell that initializes 'base' (cell 85c283ac) first.")
else:
    # Determine the actual training directory based on the layout
    # If 'layout' is not defined, assume 'single_root' for safety or prompt user.
    if 'layout' not in globals():
        print("Warning: 'layout' variable not found. Assuming 'single_root' for ImageDataGenerator setup.")
        current_layout = "single_root"
    else:
        current_layout = layout

    if current_layout == "provided_split":
        train_dir_path = next((p for p in Path(base).iterdir() if p.name.lower() == "train"), None)
    else:
        train_dir_path = base

    if train_dir_path is None:
        print("Error: Could not determine the training directory for ImageDataGenerator.")
    else:
        print(f"Using training directory: {train_dir_path}")
        datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.1,
            height_shift_range=0.1,
            shear_range=0.1,
            zoom_range=0.1,
            horizontal_flip=True,
            fill_mode='nearest',
            validation_split=0.2
        )
        flow_train = datagen.flow_from_directory(train_dir_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                                class_mode='sparse', subset='training', seed=SEED)
        flow_val = datagen.flow_from_directory(train_dir_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                              class_mode='sparse', subset='validation', seed=SEED)

        # Ensure num_classes and build_variant are defined
        if 'num_classes' not in globals() or 'build_variant' not in globals():
            print("Error: 'num_classes' or 'build_variant' not defined. Please run preceding model definition cells.")
        else:
            model_aug = build_variant(num_classes)  # Using your improved variant
            print("\nStarting augmented training...")
            hist_aug = model_aug.fit(flow_train, validation_data=flow_val, epochs=8, verbose=2)
            if 'plot_curves' in globals():
                plot_curves(hist_aug, title='Augmented training')
            else:
                print("Warning: 'plot_curves' function not found. Cannot plot training history.")

Error: 'base' is not defined. Please run the cell that initializes 'base' (cell 85c283ac) first.


In [6]:
# To-Do: build an ImageDataGenerator pipeline (Option A)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Ensure train_dir is defined from previous cells (e.g., cell 197e5ef5)
if 'base' not in globals():
    print("Error: 'base' is not defined. Please run the cell that initializes 'base' (cell 85c283ac) first.")
else:
    # Determine the actual training directory based on the layout
    if layout == "provided_split":
        train_dir_path = next((p for p in Path(base).iterdir() if p.name.lower() == "train"), None)
    else:
        train_dir_path = base

    if train_dir_path is None:
        print("Error: Could not determine the training directory.")
    else:
        datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=20,
            width_shift_range=0.1,
            height_shift_range=0.1,
            shear_range=0.1,
            zoom_range=0.1,
            horizontal_flip=True,
            fill_mode='nearest',
            validation_split=0.2
        )
        flow_train = datagen.flow_from_directory(train_dir_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                                class_mode='sparse', subset='training', seed=SEED)
        flow_val = datagen.flow_from_directory(train_dir_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                              class_mode='sparse', subset='validation', seed=SEED)

        model_aug = build_variant(num_classes)  # Using your improved variant
        hist_aug = model_aug.fit(flow_train, validation_data=flow_val, epochs=8, verbose=2)
        plot_curves(hist_aug, title='Augmented training')

Error: 'base' is not defined. Please run the cell that initializes 'base' (cell 85c283ac) first.


# XP Exercises: Flower Classification using CNN

This is a guided notebook for the exercises on the platform. Cells marked **PREFILLED** are for execution only. Cells marked **To-Do** require your action. When a written answer is required, the **To-Do** appears inside a markdown cell. When code is required, the **To-Do** appears inside a code cell as comments.

Learning points appear only for key concepts that unlock intuition or transfer to other ML topics.


## What you will learn
- Building a CNN for multi class image classification
- Data loading and preprocessing with `image_dataset_from_directory`
- Image visualization techniques
- Model architecture design, compilation, and training
- Evaluating model performance with accuracy and loss plots


## What you will create
A CNN model that classifies 14 flower species.
All parts form one continuous exercise. Work through them sequentially.


## Dataset
**As stated in the exercises**  
Flower classification with 14 classes. Images are organized in class folders. A training and validation split may be provided. Images are resized to 256x256 in this notebook.

**PREFILLED info**  
This notebook expects the provided zip file to be available. The code below extracts it and locates the dataset root automatically.


In [7]:
# PREFILLED: just execute
import os, sys, zipfile, shutil, glob, math, json, random
from pathlib import Path

DATA_ZIP = Path("./Flower Classification.zip")
EXTRACT_DIR = Path("./data/flower_data")

# Clean extract dir if re-running
if EXTRACT_DIR.exists():
    pass  # avoid deleting in case you added files; delete manually if needed
else:
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

# Extract if a zip is present and not already extracted
if DATA_ZIP.exists():
    # Heuristically decide to extract once
    marker = EXTRACT_DIR / ".extracted"
    if not marker.exists():
        with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
            zf.extractall(EXTRACT_DIR)
        marker.write_text("ok")
        print("Extracted:", DATA_ZIP.name, "->", EXTRACT_DIR)
    else:
        print("Already extracted. Skipping.")
else:
    print("Zip file not found at", DATA_ZIP)

# Find candidate dataset roots: a dir with >= 10 subdirs assumed as classes, or contains train/val
def list_dirs(p):
    return [d for d in Path(p).iterdir() if d.is_dir()]

candidates = []
for root, dirs, files in os.walk(EXTRACT_DIR):
    if len([d for d in Path(root).iterdir() if Path(d).is_dir()]) >= 10:
        candidates.append(Path(root))
    if "train" in [d.name.lower() for d in list_dirs(root)] and "val" in [d.name.lower() for d in list_dirs(root)]:
        candidates.append(Path(root))

candidates = sorted(set(candidates))
print("Candidate dataset roots:", [str(c) for c in candidates][:5])

Zip file not found at Flower Classification.zip
Candidate dataset roots: []


## Part 1. Data exploration and visualization

**As stated in the exercises**  
Load the dataset using `image_dataset_from_directory`. Print number of images per class. Modify `visualize_images` to show a 3x3 grid for each class with the class name as the grid title. Analyze challenges you anticipate when classifying the flowers such as similar colors or shapes and intra class variation.


**Guidance**  
If a `train` or `val` folder exists, use them. Otherwise create a split from a single root with `validation_split` and `subset`. Images are resized to 256x256 RGB.


> **IMPORTANT:** we fix a low resultion for images in IMG_SIZE=(32,32) for faster training, however you can change it if you want to test out other resolutions

In [8]:
# PREFILLED: just execute
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = (32, 32)
BATCH_SIZE = 32
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

def detect_layout(root: Path):
    root = Path(root)
    sub = [d.name.lower() for d in root.iterdir() if d.is_dir()]
    if "train" in sub and "val" in sub:
        return "provided_split", root
    return "single_root", root

# Choose a root
if 'candidates' in globals() and len(candidates) > 0:
    DS_ROOT = candidates[0]
else:
    DS_ROOT = EXTRACT_DIR  # fallback

layout, base = detect_layout(DS_ROOT)
print("Layout:", layout, "Base:", base)

Layout: single_root Base: data/flower_data


In [9]:
# PREFILLED: just execute
if layout == "provided_split":
    train_dir = next((p for p in base.iterdir() if p.name.lower()=="train"))
    val_dir   = next((p for p in base.iterdir() if p.name.lower()=="val"))
    train_ds = tf.keras.utils.image_dataset_from_directory(
        train_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        val_dir, image_size=IMG_SIZE, batch_size=BATCH_SIZE, seed=SEED, label_mode="int"
    )
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="training", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        base, validation_split=0.2, subset="validation", seed=SEED,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="int"
    )

class_names = train_ds.class_names
num_classes = len(class_names)
print("Classes:", num_classes, class_names)

# Cache and prefetch
def prepare(ds):
    return ds.cache().prefetch(AUTOTUNE)

train_ds = prepare(train_ds)
val_ds = prepare(val_ds)

Found 0 files belonging to 0 classes.
Using 0 files for training.


ValueError: No images found in directory data/flower_data. Allowed formats: ('.bmp', '.gif', '.jpeg', '.jpg', '.png')

In [ ]:
# PREFILLED: just execute — count images per class by scanning directory
from collections import Counter
import os

def count_images_in_directory_per_class(directory, class_names):
    counts = {}
    if directory is None or not directory.exists():
        print(f"Directory {directory} does not exist or is not specified.")
        return {}
    for cls_name in class_names:
        class_path = directory / cls_name
        if class_path.is_dir():
            img_count = sum(1 for p in class_path.rglob("*") if p.suffix.lower() in {".jpg",".jpeg",".png",".bmp",".gif"})
            counts[cls_name] = img_count
        else:
            # This case might happen if a class exists in train_ds.class_names but not in this specific directory (e.g., val_dir if it's sparse)
            counts[cls_name] = 0
    return counts

print("--- Training Set Image Counts per Class ---")
train_counts = count_images_in_directory_per_class(train_dir, class_names)
for cls_name, count in train_counts.items():
    print(f"{cls_name}: {count} images")

print("\n--- Validation Set Image Counts per Class ---")
val_counts = count_images_in_directory_per_class(val_dir, class_names)
for cls_name, count in val_counts.items():
    print(f"{cls_name}: {count} images")

In [1]:
import matplotlib.pyplot as plt

def visualize_images(dataset, class_names, per_class=9):
    class_images = {name: [] for name in class_names}
    images_collected = 0

    # Iterate through batches and collect images until 'per_class' images are gathered for each class
    for images, labels in dataset:
        for i in range(images.shape[0]):
            label_idx = int(labels[i])
            class_name = class_names[label_idx]
            if len(class_images[class_name]) < per_class:
                class_images[class_name].append(images[i].numpy())
                images_collected += 1

        # Stop if we have enough images for all classes
        if all(len(v) == per_class for v in class_images.values()):
            break

    # Plot images for each class
    for class_name, images in class_images.items():
        if not images:
            continue

        plt.figure(figsize=(8, 8))
        plt.suptitle(class_name, fontsize=16)
        for i in range(min(per_class, len(images))):
            plt.subplot(3, 3, i + 1)
            plt.imshow(images[i].astype("uint8"))
            plt.axis("off")
        plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent suptitle from overlapping
        plt.show()

In [4]:
# To-Do: Call visualize_images on your dataset

# Ensure train_ds and class_names are defined from previous cells (e.g., cell 197e5ef5)
if 'train_ds' not in locals() and 'train_ds' not in globals():
    print("Error: 'train_ds' is not defined. Please run the cell that initializes 'train_ds' and 'class_names' (cell 197e5ef5) first.")
elif 'class_names' not in locals() and 'class_names' not in globals():
    print("Error: 'class_names' is not defined. Please run the cell that initializes 'train_ds' and 'class_names' (cell 197e5ef5) first.")
else:
    visualize_images(train_ds, class_names)

Error: 'train_ds' is not defined. Please run the cell that initializes 'train_ds' and 'class_names' (cell 197e5ef5) first.


**To-Do:** After you implement `visualize_images`, run it on a small subset to verify class distributions visually.


**To-Do (written):** Analyze expected challenges for classification in 4 to 6 sentences. Mention similar color palettes across species, intra class variation due to background and lighting, and class imbalance if present.


**Learning point**  
Vision models learn features from texture, color, and shape. Dataset bias and imbalance can dominate results without careful preprocessing and evaluation.


## Part 2. Model architecture design

**As stated in the exercises**  
Start from the provided model. Experiment with the number of convolutional layers, filters, kernel sizes, max pooling layers. Try different dense layers and dropout. Consider Batch Normalization. Justify your architectural choices.


In [ ]:
# PREFILLED: just execute — baseline model scaffold
from tensorflow.keras import models

def build_baseline(num_classes):
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255),  # safety if datasets were not normalized
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

baseline = build_baseline(num_classes)
baseline.summary()

In [ ]:
# To-Do: create an improved architecture variant
# Suggestions:
# - Add BatchNormalization after Conv2D or Dense
# - Try kernel sizes 5x5 in early layers
# - Increase filters progressively 32->64->128->256
# - Adjust Dropout to 0.4

def build_variant(num_classes):
    model = models.Sequential([
        layers.Input(shape=(*IMG_SIZE, 3)),
        layers.Rescaling(1./255), # Normalize input
        layers.Conv2D(32, 5, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Conv2D(128, 3, padding="same", activation="relu"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),
        layers.Flatten(),
        layers.Dense(256, activation="relu"), # Increased dense layer units
        layers.BatchNormalization(),
        layers.Dropout(0.4), # Adjusted dropout rate
        layers.Dense(num_classes, activation="softmax")
    ])
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model

model_variant = build_variant(num_classes)
model_variant.summary()

**To-Do (written):** Justify your chosen layers and parameters in 4 to 6 sentences. Refer to receptive field growth, normalization stabilizing training, and dropout for regularization.


## Part 3. Hyperparameter tuning

**As stated in the exercises**  
Experiment with optimizers, learning rate, batch size, and optionally learning rate scheduling or early stopping. Track experiments and results. Report the best combination.


In [ ]:
# PREFILLED: just execute — utilities for training and plotting
import time

def fit_model(model, train_ds, val_ds, epochs=5, callbacks=None):
    t0 = time.time()
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs, callbacks=callbacks, verbose=2)
    dt = time.time() - t0
    return history, dt

def plot_curves(history, title="Training"):
    plt.figure(figsize=(6,4))
    plt.plot(history.history.get("accuracy", []), label="acc")
    plt.plot(history.history.get("val_accuracy", []), label="val_acc")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.tight_layout(); plt.show()
    plt.figure(figsize=(6,4))
    plt.plot(history.history.get("loss", []), label="loss")
    plt.plot(history.history.get("val_loss", []), label="val_loss")
    plt.title(title); plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# To-Do: run a few experiments
# Example search space
opts = [
  ("adam", 1e-3, 32),
  ("adam", 5e-4, 32),
  ("rmsprop", 1e-3, 32),
  ("sgd", 1e-2, 64),
]
results = []
for opt_name, lr, batch in opts:
    # rebuild model each time
    model = build_variant(num_classes)  # Using the improved variant
    if opt_name == "adam":
        optimizer = tf.keras.optimizers.Adam(lr)
    elif opt_name == "rmsprop":
        optimizer = tf.keras.optimizers.RMSprop(lr)
    else:
        optimizer = tf.keras.optimizers.SGD(lr, momentum=0.9, nesterov=True)
    model.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    cb = [tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)]
    hist, dur = fit_model(model, train_ds, val_ds, epochs=8, callbacks=cb)
    best_val = max(hist.history["val_accuracy"])
    results.append({"opt": opt_name, "lr": lr, "batch": batch, "best_val_acc": float(best_val), "time_s": round(dur,1)})
results

**To-Do (written):** Report the best hyperparameters you found and briefly explain why they might work well for this dataset.


## Part 4. Data augmentation

**As stated in the exercises**  
Implement data augmentation using `ImageDataGenerator`. Explore rotation, flipping, zooming, shifting, and shearing. Determine which augmentations help most and explain why.


**Guidance**  
Since we used `image_dataset_from_directory` above, you can either:  
Option A. Rebuild input using `ImageDataGenerator.flow_from_directory` on the training directory.  
Option B. Keep the tf.data pipeline and apply Keras preprocessing layers such as `RandomFlip`, `RandomRotation`.  
The exercises asks for `ImageDataGenerator`, so Option A shows that path.


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from pathlib import Path

# Ensure 'base' is defined from previous cells (e.g., cell 85c283ac)
if 'base' not in globals():
    print("Error: 'base' is not defined. Please run the cell that initializes 'base' (cell 85c283ac) first.")
else:
    # Determine the actual training directory based on the layout
    # If 'layout' is not defined, assume 'single_root' for safety or prompt user.
    if 'layout' not in globals():
        print("Warning: 'layout' variable not found. Assuming 'single_root' for ImageDataGenerator setup.")
        current_layout = "single_root"
    else:
        current_layout = layout

    train_dir_path = None
    if current_layout == "provided_split":
        # In 'provided_split' layout, 'base' points to the root containing 'train' and 'val' subdirectories
        train_dir_candidate = next((p for p in Path(base).iterdir() if p.name.lower() == "train"), None)
        if train_dir_candidate and train_dir_candidate.is_dir():
            train_dir_path = train_dir_candidate
    else: # current_layout == "single_root"
        # In 'single_root' layout, 'base' is the training directory itself
        if Path(base).is_dir():
            train_dir_path = Path(base)

    if train_dir_path is None:
        print("Error: Could not determine the training directory for ImageDataGenerator. Please check 'base' and 'layout'.")
    else:
        print(f"Using training directory for ImageDataGenerator: {train_dir_path}")

        # Ensure IMG_SIZE, BATCH_SIZE, SEED are defined
        if 'IMG_SIZE' not in globals() or 'BATCH_SIZE' not in globals() or 'SEED' not in globals():
            print("Error: IMG_SIZE, BATCH_SIZE, or SEED are not defined. Please run preceding configuration cells.")
        else:
            datagen = ImageDataGenerator(
                rescale=1./255,
                rotation_range=20,
                width_shift_range=0.1,
                height_shift_range=0.1,
                shear_range=0.1,
                zoom_range=0.1,
                horizontal_flip=True,
                fill_mode='nearest',
                validation_split=0.2
            )
            flow_train = datagen.flow_from_directory(train_dir_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                                    class_mode='sparse', subset='training', seed=SEED)
            flow_val = datagen.flow_from_directory(train_dir_path, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
                                                  class_mode='sparse', subset='validation', seed=SEED)

            # Ensure num_classes, build_variant, and plot_curves are defined
            if 'num_classes' not in globals() or 'build_variant' not in globals():
                print("Error: 'num_classes' or 'build_variant' not defined. Please run preceding model definition cells.")
            else:
                model_aug = build_variant(num_classes)  # Using your improved variant
                print("\nStarting augmented training...")
                # Using 8 epochs as per the commented example, adjust as needed
                hist_aug = model_aug.fit(flow_train, validation_data=flow_val, epochs=8, verbose=2)
                if 'plot_curves' in globals():
                    plot_curves(hist_aug, title='Augmented training')
                else:
                    print("Warning: 'plot_curves' function not found. Cannot plot training history.")

**Learning point**  
Augmentation encodes invariances like rotation and translation. It increases effective sample diversity which often reduces overfitting.


## Part 5. Performance evaluation and analysis

**As stated in the exercises**  
Plot training and validation curves. Compute precision, recall, F1, and a confusion matrix. Visualize predictions on a test set and analyze misclassifications.


In [ ]:
# PREFILLED: just execute — helpers for evaluation on a dataset
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

def collect_preds(model, ds):
    y_true = []
    y_prob = []
    for xb, yb in ds:
        pr = model.predict(xb, verbose=0)
        y_prob.append(pr)
        y_true.append(yb.numpy())
    y_true = np.concatenate(y_true)
    y_prob = np.concatenate(y_prob)
    if y_prob.ndim == 2 and y_prob.shape[1] > 1:
        y_pred = y_prob.argmax(axis=1)
    else:
        y_pred = (y_prob.ravel() >= 0.5).astype(int)
    return y_true, y_pred, y_prob

def plot_confusion(cm, labels):
    plt.figure(figsize=(6,6))
    plt.imshow(cm)
    plt.title("Confusion matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    ticks = np.arange(len(labels))
    plt.xticks(ticks, labels, rotation=90)
    plt.yticks(ticks, labels)
    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Assuming 'model_aug' is your best performing model from the augmented training
# If 'model_aug' is not defined, fall back to 'model_variant' or 'baseline'
if 'model_aug' in globals():
    best_model = model_aug
elif 'model_variant' in globals():
    best_model = model_variant
elif 'baseline' in globals():
    best_model = baseline
else:
    print("Error: No model (model_aug, model_variant, or baseline) found. Please train a model first.")
    best_model = None

if best_model and 'val_ds' in globals() and 'class_names' in globals() and 'collect_preds' in globals() and 'plot_confusion' in globals():
    print("Evaluating best model on validation dataset...")
    y_true, y_pred, y_prob = collect_preds(best_model, val_ds)
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    cm = confusion_matrix(y_true, y_pred)
    print("\nConfusion Matrix:")
    print(cm)
    plot_confusion(cm, class_names)
else:
    print("Error: Required variables (best_model, val_ds, class_names, collect_preds, plot_confusion) are not all defined. Please ensure preceding cells are executed.")

In [ ]:
import random
import matplotlib.pyplot as plt

# Ensure required variables are defined
if 'best_model' not in globals():
    print("Error: 'best_model' is not defined. Please ensure a model has been trained and assigned to 'best_model'.")
elif 'val_ds' not in globals():
    print("Error: 'val_ds' is not defined. Please run the data loading cells first.")
elif 'class_names' not in globals():
    print("Error: 'class_names' is not defined. Please run the data loading cells first.")
else:
    print("Visualizing predictions...")
    take = 12
    try:
        # Convert val_ds to an unbatched dataset to take individual images, then re-batch for prediction
        unbatched_val_ds = val_ds.unbatch()
        # Take a sample of 'take' images
        sampled_ds = unbatched_val_ds.shuffle(buffer_size=1000).take(take).batch(take)
        imgs, labels = next(iter(sampled_ds))

        probs = best_model.predict(imgs, verbose=0)
        preds = probs.argmax(axis=1)

        plt.figure(figsize=(10, 10))
        for i in range(take):
            plt.subplot(3, 4, i + 1)
            # Ensure image is in displayable format (e.g., scale from 0-1 to 0-255 if needed)
            # Assuming model expects 0-1, so `imgs[i].numpy()` is fine for display
            plt.imshow(imgs[i].numpy().astype('uint8'))
            true_label = class_names[int(labels[i])]
            pred_label = class_names[int(preds[i])]
            color = "green" if true_label == pred_label else "red"
            t = f"True: {true_label}\nPred: {pred_label}"
            plt.title(t, color=color, fontsize=8)
            plt.axis("off")
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"An error occurred during visualization: {e}. Make sure 'val_ds' is properly loaded and 'best_model' is trained.")

**To-Do (written):** Identify classes that your model finds difficult. Explain possible causes such as similar morphology or color, or small sample counts.


## Part 6. Model saving and deployment (optional)

**As stated in the exercises**  
Save your trained model in `.h5` or SavedModel format. Optionally consider web or cloud deployment.


In [ ]:
from pathlib import Path

# Ensure 'best_model' is defined from previous evaluation or training steps
if 'best_model' not in globals():
    print("Error: 'best_model' is not defined. Please ensure a model has been trained and assigned to 'best_model'.")
else:
    # Define the directory for saving models
    save_dir = Path("./data")
    save_dir.mkdir(parents=True, exist_ok=True)

    # Save the model in SavedModel format (as a directory)
    model_saved_path_dir = save_dir / "flower_cnn_savedmodel"
    best_model.save(model_saved_path_dir)
    print(f"Model saved to: {model_saved_path_dir.resolve()}")

    # Save the model in H5 format (as a single file)
    model_saved_path_h5 = save_dir / "flower_cnn.h5"
    best_model.save(model_saved_path_h5)
    print(f"Model saved to: {model_saved_path_h5.resolve()}")